In [1]:
import os, sys, re, json, hashlib
from pathlib import Path
from datetime import datetime

NORTHSTAR = "app-store-t26-qa"
NOTEBOOK_PATH = "notebooks/qa/49-app-store-t26-qa.ipynb"
TOWER = 26
TOWER_NAME = "Tower of App Store & Lifecycle"
MIN_SCORE = 70
BROWSER_ROOT = Path("/home/phuc/projects/solace-browser")
SRC = BROWSER_ROOT / "src"
WEB = BROWSER_ROOT / "web"
TESTS = BROWSER_ROOT / "tests"
results = []

def record(probe_id, name, passed, detail=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status, "detail": detail})
    print(f"{status}: {name}" + (f" -- {detail}" if detail else ""))

def read_file(path):
    p = Path(path)
    return p.read_text(encoding='utf-8', errors='replace') if p.exists() else ""

print(f"Tower {TOWER}: {TOWER_NAME}")
assert BROWSER_ROOT.exists()

Tower 26: Tower of App Store & Lifecycle


In [2]:
# Cell 1: App Store HTML page structure
print("=== App Store Page ===")

app_store_html = read_file(WEB / "app-store.html")

# P1: App store page exists with proper title
record("T26-P01", "app-store.html exists with App Store title",
       "App Store" in app_store_html and "<title>" in app_store_html,
       "Page title contains 'App Store'")

# P2: App store has search input
record("T26-P02", "App store has search input for filtering apps",
       'id="app-search"' in app_store_html and 'type="search"' in app_store_html,
       "Search input with id=app-search")

# P3: App store has categories section
record("T26-P03", "App store has categories section",
       'id="app-store-categories"' in app_store_html,
       "Categories container div present")

# P4: App store references /api/apps endpoint
record("T26-P04", "App store references /api/apps endpoint for loading apps",
       "/api/apps" in app_store_html,
       "Dynamic app loading from API")

# P5: App store has proposal/suggestion form
record("T26-P05", "App store has app proposal submission form",
       'id="app-proposal-form"' in app_store_html,
       "Community can suggest new apps")

# P6: App store has i18n data attributes
record("T26-P06", "App store page has i18n data-i18n attributes",
       'data-i18n=' in app_store_html,
       "Internationalization support")

# P7: App store has CSP meta tag
record("T26-P07", "App store has Content-Security-Policy meta tag",
       'Content-Security-Policy' in app_store_html,
       "Security headers present")

=== App Store Page ===
PASS: app-store.html exists with App Store title -- Page title contains 'App Store'
PASS: App store has search input for filtering apps -- Search input with id=app-search
PASS: App store has categories section -- Categories container div present
PASS: App store references /api/apps endpoint for loading apps -- Dynamic app loading from API
PASS: App store has app proposal submission form -- Community can suggest new apps
PASS: App store page has i18n data-i18n attributes -- Internationalization support
PASS: App store has Content-Security-Policy meta tag -- Security headers present


In [3]:
# Cell 2: Create App page + App Detail page
print("=== Create App + App Detail ===")

create_app_html = read_file(WEB / "create-app.html")

# P8: create-app.html exists
record("T26-P08", "create-app.html page exists",
       len(create_app_html) > 100,
       "Create your own app page")

# P9: Create app page has step progress (funnel: Describe -> Preview -> Test -> Submit)
record("T26-P09", "Create app page has step progress flow",
       'ca-progress' in create_app_html or 'ca-dot' in create_app_html,
       "Step-by-step wizard for app creation")

# P10: Create app page has Yinyang mention (AI-powered)
record("T26-P10", "Create app page references Yinyang AI assistant",
       'yinyang' in create_app_html.lower() or 'Yinyang' in create_app_html,
       "AI turns description into working recipe")

# P11: app-detail.html exists
app_detail_html = read_file(WEB / "app-detail.html")
record("T26-P11", "app-detail.html page exists for app dashboard",
       len(app_detail_html) > 100 and "App Dashboard" in app_detail_html,
       "App detail/dashboard page")

# P12: App detail has back navigation
record("T26-P12", "App detail page has back navigation link",
       'ad-back' in app_detail_html,
       "Back to catalog navigation")

# P13: App detail has runs chart section
record("T26-P13", "App detail page has runs chart visualization",
       'ad-runs-chart' in app_detail_html,
       "Execution history chart")

=== Create App + App Detail ===
PASS: create-app.html page exists -- Create your own app page
PASS: Create app page has step progress flow -- Step-by-step wizard for app creation
PASS: Create app page references Yinyang AI assistant -- AI turns description into working recipe
PASS: app-detail.html page exists for app dashboard -- App detail/dashboard page
PASS: App detail page has back navigation link -- Back to catalog navigation
PASS: App detail page has runs chart visualization -- Execution history chart


In [4]:
# Cell 3: App Store backend + official catalog
print("=== App Store Backend ===")

# P14: App store backend module exists
backend_py = SRC / "app_store" / "backend.py"
backend_content = read_file(backend_py)
record("T26-P14", "App store backend module exists",
       len(backend_content) > 100,
       str(backend_py))

# P15: Backend has proposal validation
record("T26-P15", "App store backend has proposal validation",
       'AppStoreProposalValidationError' in backend_content,
       "Proposals validated before acceptance")

# P16: Official store catalog JSON exists
official_store = BROWSER_ROOT / "data" / "default" / "app-store" / "official-store.json"
record("T26-P16", "Official store catalog JSON exists",
       official_store.exists(),
       str(official_store))

# P17: Official store has apps array with categories
store_content = read_file(official_store)
if store_content:
    store_data = json.loads(store_content)
    apps = store_data.get("apps", [])
    categories = set(a.get("category", "") for a in apps)
    record("T26-P17", "Official store has multiple apps with categories",
           len(apps) >= 5 and len(categories) >= 3,
           f"apps={len(apps)}, categories={sorted(categories)}")
else:
    record("T26-P17", "Official store has multiple apps with categories", False, "File not found")

# P18: Apps have scopes defined
if store_content:
    apps_with_scopes = [a for a in apps if a.get("scopes") and len(a.get("scopes", [])) > 0]
    record("T26-P18", "Apps in catalog have OAuth3 scopes defined",
           len(apps_with_scopes) >= 3,
           f"{len(apps_with_scopes)}/{len(apps)} apps have scopes")
else:
    record("T26-P18", "Apps in catalog have OAuth3 scopes defined", False, "File not found")

=== App Store Backend ===
PASS: App store backend module exists -- /home/phuc/projects/solace-browser/src/app_store/backend.py
PASS: App store backend has proposal validation -- Proposals validated before acceptance
PASS: Official store catalog JSON exists -- /home/phuc/projects/solace-browser/data/default/app-store/official-store.json
PASS: Official store has multiple apps with categories -- apps=18, categories=['communications', 'engineering', 'productivity', 'sales', 'shopping', 'social']
PASS: Apps in catalog have OAuth3 scopes defined -- 18/18 apps have scopes


In [5]:
# Cell 4: Server API routes for app store
print("=== Server API Routes ===")

server_py = read_file(WEB / "server.py")

# P19: Server has /api/apps route
record("T26-P19", "Server has /api/apps list route",
       '/api/apps' in server_py and 'APP_LIST_ROUTE' in server_py,
       "App catalog API endpoint")

# P20: Server has /api/apps/{appId} detail route
record("T26-P20", "Server has /api/apps/{appId} detail route",
       'app_detail_match' in server_py or '/api/apps/' in server_py,
       "Individual app detail endpoint")

# P21: Server has /api/app-store/sync route
record("T26-P21", "Server has /api/app-store/sync route",
       '/api/app-store/sync' in server_py,
       "App store synchronization endpoint")

# P22: Server has /api/app-store/proposals route
record("T26-P22", "Server has /api/app-store/proposals route",
       '/api/app-store/proposals' in server_py,
       "Community proposal endpoint")

# P23: Server has /api/apps/installed route
record("T26-P23", "Server has /api/apps/installed route",
       '/api/apps/installed' in server_py,
       "Installed apps listing endpoint")

# P24: App runner module exists
app_runner = SRC / "apps" / "runner.py"
record("T26-P24", "App runner module exists for executing app recipes",
       app_runner.exists(),
       str(app_runner))

# P25: Proposed apps dev file exists
proposed = BROWSER_ROOT / "data" / "default" / "app-store" / "proposed-apps-dev.jsonl"
record("T26-P25", "Proposed apps JSONL file exists for community suggestions",
       proposed.exists(),
       str(proposed))

=== Server API Routes ===
PASS: Server has /api/apps list route -- App catalog API endpoint
PASS: Server has /api/apps/{appId} detail route -- Individual app detail endpoint
PASS: Server has /api/app-store/sync route -- App store synchronization endpoint
PASS: Server has /api/app-store/proposals route -- Community proposal endpoint
PASS: Server has /api/apps/installed route -- Installed apps listing endpoint
PASS: App runner module exists for executing app recipes -- /home/phuc/projects/solace-browser/src/apps/runner.py
PASS: Proposed apps JSONL file exists for community suggestions -- /home/phuc/projects/solace-browser/data/default/app-store/proposed-apps-dev.jsonl


In [6]:
# Evidence summary cell
passed = len([r for r in results if r["status"] == "PASS"])
failed = len([r for r in results if r["status"] == "FAIL"])
total = len(results)
score = round(passed / total * 100, 1) if total > 0 else 0
finding_rate = round(failed / total * 100, 1) if total > 0 else 0
grade = "A+" if score >= 95 else "A" if score >= 90 else "B+" if score >= 85 else "B" if score >= 80 else "C" if score >= 70 else "D" if score >= 60 else "F"
summary = {"surface": NORTHSTAR, "tower": TOWER, "tower_name": TOWER_NAME, "timestamp": datetime.now().isoformat(), "total_probes": total, "passed": passed, "failed": failed, "score": score, "grade": grade, "finding_rate": finding_rate, "converged": finding_rate < 5, "findings": [r for r in results if r["status"] == "FAIL"], "evidence_hash": hashlib.sha256(json.dumps(results, sort_keys=True).encode()).hexdigest()[:16]}
print("=" * 60)
print(f"TOWER {TOWER}: {TOWER_NAME}")
print("=" * 60)
print(f"Probes: {total} | Passed: {passed} | Failed: {failed}")
print(f"Score: {score}% | Grade: {grade} | Finding rate: {finding_rate}%")
print(f"Converged: {'YES' if summary['converged'] else 'NO'}")
print(f"Evidence hash: {summary['evidence_hash']}")
if summary["findings"]:
    print("\nFINDINGS:")
    for f in summary["findings"]:
        print(f"  [{f['id']}] {f['name']}: {f.get('detail', '')}")
print()
print(json.dumps(summary, indent=2))

TOWER 26: Tower of App Store & Lifecycle
Probes: 25 | Passed: 25 | Failed: 0
Score: 100.0% | Grade: A+ | Finding rate: 0.0%
Converged: YES
Evidence hash: efdbf43645b585bb

{
  "surface": "app-store-t26-qa",
  "tower": 26,
  "tower_name": "Tower of App Store & Lifecycle",
  "timestamp": "2026-03-06T11:29:14.058234",
  "total_probes": 25,
  "passed": 25,
  "failed": 0,
  "score": 100.0,
  "grade": "A+",
  "finding_rate": 0.0,
  "converged": true,
  "findings": [],
  "evidence_hash": "efdbf43645b585bb"
}
